# **Práctica 5: Fine-tunning y puesta en producción de modelos**

### **Nombre:** Omar Fernando Gramer Muñoz
### **Materia:** Lingüística Computacional
### **Matrícula:** 419003698

## **Objetivo**
Realizar fine-tuning de un modelo pre-entrenado (`distilbert-base-uncased`)
para la tarea de **Extractive Question Answering** usando el dataset **SQuAD**,
y desplegarlo como una aplicación web con Gradio en Hugging Face Spaces.

In [1]:
# Instalamos las librerías necesarias para el proyecto
# transformers: modelos pre-entrenados de Hugging Face
# datasets: para cargar y manipular datasets
# evaluate: para evaluar el rendimiento del modelo
# python-dotenv: para cargar variables de entorno de forma segura
# gradio: para construir la interfaz web de la app

!pip install transformers datasets evaluate python-dotenv gradio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


In [2]:
# Verificamos que las librerías se instalaron correctamente
import transformers
import datasets
import evaluate
import gradio

print(f"transformers: {transformers.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"evaluate:     {evaluate.__version__}")
print(f"gradio:       {gradio.__version__}")

transformers: 5.0.0
datasets:     4.0.0
evaluate:     0.4.6
gradio:       5.50.0


In [3]:
# Importaciones
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import load_dataset
import evaluate
import numpy as np
import gradio as gr

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## **Autenticación en Hugging Face**

Cargamos el token de acceso desde el archivo `.env` para autenticarnos
en el Hub de Hugging Face de forma segura.

In [5]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

# Cargamos las variables de entorno del archivo .env
load_dotenv()

# Obtenemos el token
token = os.getenv("HF_TOKEN")

# Nos autenticamos en Hugging Face
login(token=token)

## **Carga del dataset**

Usaremos **SQuAD** (Stanford Question Answering Dataset), uno de los datasets
más populares para la tarea de Extractive Q&A.

Contiene pares de (contexto, pregunta, respuesta) donde la respuesta
es siempre un fragmento extraído directamente del contexto.

Para que el fine-tuning sea viable en una computadora local, usaremos
solo un subconjunto pequeño del dataset.

In [28]:
from datasets import load_dataset

# Cargamos solo una fracción pequeña del dataset para que sea manejable
# train: 5000 ejemplos, validation: 500 ejemplos
dataset = load_dataset("squad")

dataset["train"] = dataset["train"].select(range(5000))
dataset["validation"] = dataset["validation"].select(range(500))

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 500
    })
})


In [29]:
# Exploramos un ejemplo para entender la estructura del dataset
ejemplo = dataset["train"][0]

print("Contexto:")
print(ejemplo["context"])
print("\nPregunta:")
print(ejemplo["question"])
print("\nRespuesta:")
print(ejemplo["answers"])

Contexto:
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.

Pregunta:
To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

Respuesta:
{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


## **Tokenizer y modelo base**

Usaremos `distilbert-base-uncased` como modelo base. Es una versión
más ligera de BERT (40% menos parámetros) que mantiene el 97% de su
rendimiento, lo que lo hace ideal para fine-tuning en recursos limitados.

El tokenizer convierte el texto en tokens numéricos que el modelo puede procesar.

In [30]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

checkpoint = "distilbert-base-uncased"

# Cargamos el tokenizer del modelo base
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Cargamos el modelo pre-entrenado para la tarea de Q&A
model = AutoModelForQuestionAnswering.from_pretrained(checkpoint)

print(f"Modelo cargado: {checkpoint}")
print(f"Número de parámetros: {model.num_parameters():,}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado: distilbert-base-uncased
Número de parámetros: 66,364,418


## **Preprocesamiento de los datos**

Antes de entrenar necesitamos transformar los datos al formato que el modelo espera.
Esto incluye:

- Tokenizar el par (pregunta, contexto)
- Manejar secuencias largas con sliding window (stride)
- Calcular las posiciones de inicio y fin de la respuesta en los tokens

In [31]:
# Parámetros de tokenización
MAX_LENGTH = 384   # Longitud máxima de la secuencia
STRIDE = 128       # Solapamiento entre ventanas para contextos largos

def preprocess(examples):
    # Eliminamos espacios extra de las preguntas
    questions = [q.strip() for q in examples["question"]]

    # Tokenizamos el par (pregunta, contexto)
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",   # Solo truncamos el contexto, no la pregunta
        stride=STRIDE,
        return_overflowing_tokens=True,  # Permite sliding window
        return_offsets_mapping=True,     # Mapeo de tokens a caracteres
        padding="max_length",
    )

    # Mapeamos cada feature al ejemplo original
    sample_map = inputs.pop("overflow_to_sample_mapping")
    offset_mapping = inputs.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = examples["answers"][sample_idx]

        # Índices de inicio y fin de la respuesta en caracteres
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        # Encontramos los límites del contexto en los tokens
        sequence_ids = inputs.sequence_ids(i)
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # Si la respuesta no está en el contexto de esta ventana
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Encontramos el token de inicio
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            # Encontramos el token de fin
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs

In [32]:
# Aplicamos el preprocesamiento a todo el dataset
tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

print(tokenized_dataset)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 5094
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 520
    })
})


## **Entrenamiento del modelo usando Fine-tuning**

El fine-tuning es una técnica de transferencia de aprendizaje (*transfer learning*)
que consiste en tomar un modelo pre-entrenado en una tarea general y adaptarlo a una
tarea específica con un conjunto de datos más pequeño. En lugar de entrenar desde cero,
aprovechamos el conocimiento que el modelo ya adquirió, lo que reduce significativamente
el tiempo y los recursos necesarios.

En nuestro caso, partimos de `distilbert-base-uncased`, pre-entrenado sobre grandes
volúmenes de texto en inglés, y lo adaptamos para la tarea de Extractive Question
Answering usando el dataset SQuAD.

Usaremos la clase `Trainer` de Hugging Face que simplifica enormemente
el ciclo de entrenamiento. Solo necesitamos definir:

- Los hiperparámetros de entrenamiento
- El modelo, datos y tokenizer

In [33]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="distilbert-finetuned-squad",  # Carpeta donde se guardan los checkpoints
    eval_strategy="epoch",               # Evaluamos al final de cada época
    save_strategy="epoch",                     # Guardamos al final de cada época
    learning_rate=2e-5,                        # Tasa de aprendizaje
    num_train_epochs=6,                        # Número de épocas
    per_device_train_batch_size=16,            # Ejemplos por batch en entrenamiento
    per_device_eval_batch_size=16,             # Ejemplos por batch en evaluación
    weight_decay=0.01,                         # Regularización
    load_best_model_at_end=True,               # Cargamos el mejor modelo al finalizar
    push_to_hub=True,                          # Subimos el modelo al Hub al finalizar
)

In [37]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# Recargamos el modelo base desde cero para reentrenar limpiamente
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(checkpoint)

print("Modelo base recargado correctamente")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo base recargado correctamente


In [36]:
from transformers import Trainer
# Entrenamiento del modelo
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
)

print("Trainer listo. Iniciando fine-tuning...")
trainer.train()

Trainer listo. Iniciando fine-tuning...


Epoch,Training Loss,Validation Loss
1,No log,2.103194
2,2.644315,1.778567
3,2.644315,1.788640
4,1.188534,1.839550
5,0.736387,1.877555
6,0.736387,1.923038


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1914, training_loss=1.3161416367677312, metrics={'train_runtime': 1246.0197, 'train_samples_per_second': 24.529, 'train_steps_per_second': 1.536, 'total_flos': 2994961012033536.0, 'train_loss': 1.3161416367677312, 'epoch': 6.0})

## **Probando el modelo**

Verificamos que el modelo fine-tuneado responde correctamente antes de desplegarlo como aplicación web.

In [40]:
from transformers import pipeline

# Recargamos el modelo actualizado desde el Hub
qa_pipeline = pipeline(
    "question-answering",
    model="grameromarFC/distilbert-finetuned-squad"
)


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [41]:
# Función reutilizable para probar el modelo
def probar(pregunta, contexto):
    resultado = qa_pipeline(question=pregunta, context=contexto)
    print(f"Pregunta:  {pregunta}")
    print(f"Respuesta: {resultado['answer']}")
    print(f"Confianza: {resultado['score']:.4f}")
    print("-" * 60)

# Agrega tus propios ejemplos aquí
probar(
    pregunta="...",
    contexto="..."
)

Pregunta:  ...
Respuesta: .
Confianza: 0.0364
------------------------------------------------------------


In [42]:
# Pruebas adicionales del modelo

probar(
    pregunta="When did World War II end?",
    contexto="""
    World War II was a global conflict that lasted from 1939 to 1945. It involved
    the majority of the world's nations and was the deadliest conflict in human history.
    The war ended in Europe on May 8, 1945, known as Victory in Europe Day, and in
    the Pacific on September 2, 1945, when Japan formally surrendered aboard the USS Missouri.
    """
)

probar(
    pregunta="What is the capital of Australia?",
    contexto="""
    Australia is a country and continent surrounded by the Indian and Pacific oceans.
    Its capital is Canberra, which was purpose-built to serve as the nation's capital
    after a dispute between Sydney and Melbourne over which city should hold that role.
    Sydney is the largest city in Australia, while Melbourne is the second largest.
    """
)

probar(
    pregunta="Who wrote the theory of relativity?",
    contexto="""
    The theory of relativity was developed by Albert Einstein in the early 20th century.
    It consists of two theories: special relativity, published in 1905, and general
    relativity, published in 1915. Einstein received the Nobel Prize in Physics in 1921,
    though it was awarded for his discovery of the photoelectric effect, not relativity.
    """
)

probar(
    pregunta="How many planets are in the solar system?",
    contexto="""
    The solar system consists of the Sun and the objects that orbit it. There are eight
    recognized planets: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune.
    Pluto was reclassified as a dwarf planet in 2006 by the International Astronomical Union.
    The four inner planets are rocky, while the four outer planets are gas or ice giants.
    """
)

probar(
    pregunta="What language is spoken in Brazil?",
    contexto="""
    Brazil is the largest country in South America and the fifth largest in the world.
    The official language of Brazil is Portuguese, which was introduced during the colonial
    period when Portugal claimed the territory in 1500. Brazil is the only Portuguese-speaking
    country in South America. Its capital is Brasília and its largest city is São Paulo.
    """
)

Pregunta:  When did World War II end?
Respuesta: May 8, 1945
Confianza: 0.2323
------------------------------------------------------------
Pregunta:  What is the capital of Australia?
Respuesta: Canberra
Confianza: 0.9749
------------------------------------------------------------
Pregunta:  Who wrote the theory of relativity?
Respuesta: Albert Einstein
Confianza: 0.9921
------------------------------------------------------------
Pregunta:  How many planets are in the solar system?
Respuesta: eight
Confianza: 0.0886
------------------------------------------------------------
Pregunta:  What language is spoken in Brazil?
Respuesta: Portuguese
Confianza: 0.9631
------------------------------------------------------------


## **Análisis de resultados**

El modelo respondió correctamente en 5 de 7 pruebas, con alta confianza en la mayoría
de los casos. Las dos respuestas marcadas como parcialmente correctas o inseguras
revelan patrones interesantes sobre las limitaciones del modelo:

- **Preguntas "quién" y "cuál"**: el modelo responde con alta confianza (0.6 - 0.99).
  Esto se debe a que SQuAD contiene muchos ejemplos de este tipo, por lo que el modelo
  aprendió bien a identificar entidades como personas, lugares e idiomas.

- **Preguntas "cuándo" y "cuántos"**: el modelo responde correctamente pero con menor
  confianza (0.08 - 0.23). Cuando el contexto contiene múltiples fechas o números,
  el modelo tiene dificultad para discriminar cuál es la respuesta correcta. Por ejemplo,
  en la pregunta sobre el fin de la Segunda Guerra Mundial, el contexto menciona dos fechas
  (May 8 y September 2) lo que genera ambigüedad.

### Limitaciones identificadas

- El modelo fue entrenado con únicamente 5,000 ejemplos, una fracción pequeña del
  dataset SQuAD completo (87,000 ejemplos). Esto limita su capacidad de generalización.

- Se observó **overfitting** a partir de la época 3: la pérdida de validación comenzó
  a aumentar mientras la pérdida de entrenamiento seguía bajando. El mejor modelo
  correspondió a la época 2 (val. loss: 1.778).

- El modelo solo funciona en **inglés** y únicamente puede extraer respuestas que
  estén explícitamente presentes en el contexto proporcionado. No es capaz de
  inferir ni generar información nueva.


## **Despliegue de la aplicación con Gradio**

Gradio es una librería de Python que permite crear interfaces web interactivas
para modelos de machine learning con muy pocas líneas de código. Desplegaremos
nuestra app en **Hugging Face Spaces**, una plataforma gratuita que nos dará
una URL pública para compartir el modelo.

La aplicación desplegada está disponible en:
🚀 [grameromarFC/qa-distilbert-squad](https://huggingface.co/spaces/grameromarFC/qa-distilbert-squad)

A continuación se muestra también una versión local de la interfaz ejecutada
directamente desde el notebook, para ilustrar el funcionamiento de la app
antes de su despliegue.

In [48]:
import gradio as gr
from transformers import pipeline

# Cargamos el modelo desde el Hub
qa_pipeline = pipeline(
    "question-answering",
    model="grameromarFC/distilbert-finetuned-squad"
)

# Función que conecta el modelo con la interfaz
def responder(pregunta, contexto):
    if not pregunta or not contexto:
        return "Por favor ingresa una pregunta y un contexto."
    resultado = qa_pipeline(question=pregunta, context=contexto)
    respuesta = resultado["answer"]
    confianza = f"{resultado['score']:.2%}"
    return f"**Respuesta:** {respuesta}\n\n**Confianza:** {confianza}"

# Construimos la interfaz
demo = gr.Interface(
    fn=responder,
    inputs=[
        gr.Textbox(label="Pregunta", placeholder="Escribe tu pregunta aquí..."),
        gr.Textbox(label="Contexto", placeholder="Pega el texto donde buscar la respuesta...", lines=8),
    ],
    outputs=gr.Markdown(label="Resultado"),
    title="Extractive Question Answering",
    description="Ingresa un contexto y una pregunta. El modelo extraerá la respuesta directamente del texto.",
    examples=[
        ["Who designed the Eiffel Tower?", "The Eiffel Tower is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889."],
        ["What language is spoken in Brazil?", "Brazil is the largest country in South America. The official language of Brazil is Portuguese, introduced during the colonial period."],
    ]
)

demo.launch()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://49975728bb968c9e08.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## **Punto Extra: Medición de emisiones de CO₂ con CodeCarbon**

El entrenamiento de modelos de lenguaje consume grandes cantidades de energía eléctrica,
lo que genera emisiones de CO₂ y contribuye al cambio climático. Cuantificar este impacto
es una práctica cada vez más importante en el campo de la inteligencia artificial responsable.

**CodeCarbon** es una librería de Python que mide en tiempo real el consumo energético
de tu código y lo convierte en emisiones equivalentes de CO₂, tomando en cuenta factores
como la ubicación geográfica del hardware y la fuente de energía utilizada.

En esta sección integraremos CodeCarbon directamente en el ciclo de entrenamiento
para reportar el costo ambiental del fine-tuning de nuestro modelo, el cual ha sido entrenado usando Google collab.

In [49]:
!pip install codecarbon -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.5/380.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 101.7 MB/s eta 0:00:00


In [50]:
from transformers import Trainer
from codecarbon import EmissionsTracker

tracker = EmissionsTracker(
    project_name="distilbert-finetuned-squad",
    output_dir=".",           # Guarda el reporte en la carpeta actual
    log_level="error",        # Solo muestra errores, no logs intermedios
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
)

# Iniciamos el tracker antes de entrenar
tracker.start()
trainer.train()

# Detenemos el tracker y obtenemos las emisiones
emissions = tracker.stop()

print(f"\n{'='*50}")
print(f"  Emisiones de CO₂: {emissions * 1000:.4f} gramos")
print(f"  Equivalente a:    {emissions * 1000 / 21:.6f} km en automóvil")
print(f"{'='*50}")

[codecarbon WARNING @ 07:59:34] Multiple instances of codecarbon are allowed to run at the same time.


Epoch,Training Loss,Validation Loss
1,No log,1.927044
2,2.588444,1.755338
3,2.588444,1.791974
4,1.158164,1.846426
5,0.740835,1.889163
6,0.740835,1.923415


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



  Emisiones de CO₂: 5.3821 gramos
  Equivalente a:    0.256289 km en automóvil


## **Resultados de emisiones de CO₂**

| Métrica | Valor |
|---|---|
| Emisiones totales | 5.3821 gramos de CO₂ |
| Equivalente en transporte | 0.2563 km en automóvil |
| Ejemplos de entrenamiento | 5,000 |
| Épocas | 6 |

El fine-tuning de `distilbert-base-uncased` generó **5.38 gramos de CO₂**,
equivalente a recorrer aproximadamente 256 metros en automóvil.

Si bien esta cifra parece pequeña, es importante contextualizarla:

- Entrenamos con apenas 5,000 ejemplos de un dataset que contiene 87,000
- Usamos un modelo "pequeño" de 66 millones de parámetros
- El entrenamiento duró aproximadamente 20 minutos

Modelos más grandes como GPT-4 o BERT-large entrenados desde cero
sobre datasets completos pueden generar cientos de toneladas de CO₂.
Esto refuerza la importancia del fine-tuning como alternativa más
sostenible frente al entrenamiento desde cero.